# Analisis Kualitas dan Pembersihan Data Transaksi Koperasi 2025

**Nama:** Aqila Diliansyah Putra

**Nim:** 0110225025

**Mata kuliah:** Exploratory Data Analysis (KMTI21133)  
**Pengerjaan:** Workbook analisis mandiri  
**Dataset:** `transaksi_koperasi_2025.csv`

Notebook ini disusun sebagai alur analisis yang dapat direproduksi. Hasil, jumlah baris terdampak, dan keputusan pembersihan dihitung dari file yang dijalankan. Tinjau kembali setiap aturan bisnis sebelum menggunakan hasilnya dalam laporan.

> **Integritas akademik:** notebook ini merupakan kerangka analisis yang disusun ulang, bukan jaminan lolos alat pendeteksi kemiripan. Pahami kode, validasi keputusan, dan tuliskan interpretasi akhir dengan pemahaman serta kata-katamu sendiri.

## 1. Tujuan analisis

Analisis difokuskan pada lima dimensi kualitas data: kelengkapan, keunikan, konsistensi, validitas, dan akurasi. Tahapan dilakukan berurutan agar perubahan pada satu kolom tidak menutupi masalah di tahap berikutnya.

**Prinsip:** data mentah tidak ditimpa. Semua perubahan diterapkan pada salinan dan hasil akhir disimpan terpisah.

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

NAMA_FILE = "transaksi_koperasi_2025.csv"

# Google Colab: pilih file CSV jika belum tersedia di direktori kerja.
if not Path(NAMA_FILE).exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            NAMA_FILE = next(iter(uploaded))
    except ImportError:
        pass

data_mentah = pd.read_csv(NAMA_FILE)
data = data_mentah.copy(deep=True)

print(f"Data berhasil dibaca: {data.shape[0]:,} baris × {data.shape[1]} kolom")
display(data.head())

Saving transaksi_koperasi_2025 (1).csv to transaksi_koperasi_2025 (1).csv
Data berhasil dibaca: 1,543 baris × 16 kolom


,id_transaksi,tanggal_transaksi,id_anggota,nama_produk,kategori,satuan,jumlah,harga_satuan,diskon_persen,total_bayar,metode_bayar,kasir,saldo_simpanan,berat_kg,rating_layanan,catatan_kasir
0,TRX251385,2025-10-29 11:19:00,AGT1150,Mi Instan Rebus,Makanan,porsi,6,10000,0,60000,QRIS,Rina,392000.0,2.445,3,NaN
1,TRX251333,2025-09-29 11:46:00,AGT1259,Teh Botol,Minuman,botol,3,6000,0,Rp 18.000,Saldo Koperasi,Dimas,152000.0,0.583,5,Pelanggan tetap
2,TRX251278,2025-10-25 13:18:00,AGT1145,Map Plastik,ATK,pcs,4,4000,0,16000,Saldo Koperasi,Bagus,127000.0,2.181,5,Pelanggan tetap
3,TRX251304,2025-09-09 13:45:00,AGT1240,Fotokopi A4,Alat Tulis,pcs,1,500,0,250,Tunai,Rina,1008000.0,NaN,5,NaN
4,TRX251364,2025-08-09 10:48:00,AGT1191,Map Plastik,ATK,pcs,5,4000,5,19000,QRIS,Rina,139000.0,2.376,4,Pelanggan tetap


## 2. Pemeriksaan awal

Pemeriksaan awal dipakai untuk memahami ukuran data, tipe kolom, sebaran nilai, dan kelengkapan. Pada tahap ini belum ada baris yang dihapus.

In [2]:
print("Dimensi:", data.shape)
display(data.dtypes.rename("tipe_awal").to_frame())
display(data.describe(include="all").T)
ringkasan_kosong = pd.DataFrame({
    "jumlah_kosong": data.isna().sum(),
    "persentase": (data.isna().mean() * 100).round(2)
}).sort_values("persentase", ascending=False)
display(ringkasan_kosong)

Dimensi: (1543, 16)


,tipe_awal
id_transaksi,object
tanggal_transaksi,object
id_anggota,object
nama_produk,object
kategori,object
satuan,object
jumlah,int64
harga_satuan,int64
diskon_persen,int64
total_bayar,object


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id_transaksi,1536,1493,TRX251348,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tanggal_transaksi,1543,1495,23/10/2025 09:52,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id_anggota,1543,318,AGT1041,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nama_produk,1543,18,Roti Isi,144,NaN,NaN,NaN,NaN,NaN,NaN,NaN
kategori,1543,18,Makanan,469,NaN,NaN,NaN,NaN,NaN,NaN,NaN
satuan,1543,5,porsi,508,NaN,NaN,NaN,NaN,NaN,NaN,NaN
jumlah,1543.0,NaN,NaN,NaN,4.117952,2.57904,-8.0,2.0,4.0,6.0,8.0
harga_satuan,1543.0,NaN,NaN,NaN,7891.769281,6549.147006,500.0,4000.0,6000.0,10000.0,35000.0
diskon_persen,1543.0,NaN,NaN,NaN,1.007777,6.628844,-20.0,0.0,0.0,0.0,150.0
total_bayar,1543,220,12000,79,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,jumlah_kosong,persentase
catatan_kasir,1044,67.66
berat_kg,351,22.75
saldo_simpanan,159,10.30
metode_bayar,9,0.58
id_transaksi,7,0.45
tanggal_transaksi,0,0.00
id_anggota,0,0.00
nama_produk,0,0.00
harga_satuan,0,0.00
jumlah,0,0.00


## 3. Menandai placeholder dan merapikan teks

Placeholder teks seperti tanda strip atau frasa “tidak diketahui” tidak selalu berarti hal yang sama di setiap kolom. Di sini nilai tersebut hanya dinormalisasi menjadi kosong pada kolom metode pembayaran, karena di sana nilai tersebut jelas bukan metode pembayaran yang valid. Kolom lain diperiksa terpisah.

Spasi di awal/akhir dan variasi kapitalisasi dibersihkan pada kolom teks. Data mentah tetap tersedia pada `data_mentah`.

In [3]:
# Rapikan spasi luar untuk seluruh kolom teks.
kolom_teks = data.select_dtypes(include="object").columns.tolist()
for kolom in kolom_teks:
    data[kolom] = data[kolom].map(lambda x: x.strip() if isinstance(x, str) else x)

# Placeholder metode pembayaran yang tidak menyatakan metode valid.
placeholder_metode = {"", "-", "tidak diketahui", "unknown", "n/a", "na", "null"}
data["metode_bayar"] = data["metode_bayar"].map(
    lambda x: np.nan if isinstance(x, str) and x.casefold() in placeholder_metode else x
)

# Standarisasi kapitalisasi pada kolom kategori teks tertentu.
for kolom in ["metode_bayar", "kasir"]:
    data[kolom] = data[kolom].map(
        lambda x: x.title() if isinstance(x, str) else x
    )

print("Jumlah kosong metode_bayar setelah penandaan placeholder:",
      int(data["metode_bayar"].isna().sum()))

Jumlah kosong metode_bayar setelah penandaan placeholder: 28


## 4. Konversi tipe data

Kolom tanggal dikonversi ke datetime dengan parser format campuran. Kolom pembayaran dibersihkan dari simbol mata uang dan pemisah ribuan sebelum diubah menjadi numerik. Nilai yang gagal dikonversi dibiarkan sebagai `NaN` agar dapat diperiksa.

In [4]:
data["tanggal_transaksi"] = pd.to_datetime(
    data["tanggal_transaksi"], format="mixed", dayfirst=True, errors="coerce"
)

def ubah_ke_angka_uang(nilai):
    if pd.isna(nilai):
        return np.nan
    teks = str(nilai).strip()
    teks = re.sub(r"[^0-9,.-]", "", teks)
    # Data contoh menggunakan titik sebagai pemisah ribuan.
    teks = teks.replace(".", "").replace(",", ".")
    try:
        return float(teks)
    except ValueError:
        return np.nan

data["total_bayar"] = data["total_bayar"].map(ubah_ke_angka_uang)

print(data[["tanggal_transaksi", "total_bayar"]].dtypes)
print("Tanggal gagal dibaca:", int(data["tanggal_transaksi"].isna().sum()))
print("Pembayaran gagal dibaca:", int(data["total_bayar"].isna().sum()))

tanggal_transaksi    datetime64[ns]
total_bayar                 float64
dtype: object
Tanggal gagal dibaca: 0
Pembayaran gagal dibaca: 0


## 5. Konsistensi kategori

Pemetaan berikut merupakan keputusan eksplisit berdasarkan label yang tampak sebagai variasi penulisan dari kategori yang sama. Nilai sebelum dan sesudah pemetaan ditampilkan agar keputusan dapat ditinjau. Label yang tidak tercantum tidak diubah.

In [5]:
kategori_sebelum = data["kategori"].value_counts(dropna=False)
peta_kategori = {
    "Minumam": "Minuman",
    "MINUMAN": "Minuman",
    "Minuman": "Minuman",
    "MAKANAN": "Makanan",
    "Makanan": "Makanan",
    "ATK": "ATK",
    "Alat Tulis": "ATK",
    "JASA": "Jasa",
    "Jasa": "Jasa",
    "PULSA": "Pulsa",
    "Pulsa": "Pulsa",
}
data["kategori"] = data["kategori"].replace(peta_kategori)
display(pd.DataFrame({
    "sebelum": kategori_sebelum,
    "sesudah": data["kategori"].value_counts(dropna=False)
}).fillna(0).astype(int))

,sebelum,sesudah
kategori,,
ATK,282,323
Alat Tulis,41,0
JASA,7,0
Jasa,144,151
MAKANAN,23,0
MINUMAN,21,0
Makanan,485,508
Minumam,22,0
Minuman,370,413


## 6. Pemeriksaan keunikan dan identitas transaksi

ID transaksi kosong atau berulang perlu ditandai. Pengulangan ID belum otomatis membuktikan bahwa baris merupakan transaksi duplikat: bisa terjadi salah input ID atau satu transaksi tercatat lebih dari sekali. Karena itu notebook ini membuat penanda dan tidak menghapus transaksi secara otomatis.

In [6]:
data["flag_id_kosong"] = data["id_transaksi"].isna() | data["id_transaksi"].eq("")
data["flag_id_berulang"] = data["id_transaksi"].notna() & data["id_transaksi"].duplicated(keep=False)
data["flag_duplikat_persis"] = data.duplicated(keep=False)

print("Baris dengan ID kosong:", int(data["flag_id_kosong"].sum()))
print("Baris dengan ID berulang:", int(data["flag_id_berulang"].sum()))
print("Baris yang termasuk duplikat persis:", int(data["flag_duplikat_persis"].sum()))

display(data.loc[data["flag_id_berulang"],
                 ["id_transaksi", "tanggal_transaksi", "id_anggota", "nama_produk", "total_bayar"]]
      .sort_values("id_transaksi").head(20))

Baris dengan ID kosong: 7
Baris dengan ID berulang: 85
Baris yang termasuk duplikat persis: 78


,id_transaksi,tanggal_transaksi,id_anggota,nama_produk,total_bayar
677,TRX250024,2025-11-10 12:47:00,AGT1223,Gorengan 5pcs,30000.0
663,TRX250024,2025-11-10 12:47:00,AGT1223,Gorengan 5pcs,30000.0
583,TRX250132,2025-11-05 13:17:00,AGT1010,Susu Kotak,15000.0
257,TRX250132,2025-11-05 13:17:00,AGT1010,Susu Kotak,15000.0
1109,TRX250149,2025-10-11 08:11:00,AGT1071,Buku Tulis 38lbr,11000.0
1208,TRX250149,2025-10-11 08:11:00,AGT1071,Buku Tulis 38lbr,11000.0
1279,TRX250179,2025-10-29 17:44:00,AGT1210,Teh Botol,24000.0
1167,TRX250179,2025-10-29 17:44:00,AGT1210,Teh Botol,24000.0
634,TRX250199,2025-10-31 12:48:00,AGT1041,Paket Data 3GB,35000.0
475,TRX250199,2025-10-31 12:48:00,AGT1041,Paket Data 3GB,35000.0


## 7. Validitas nilai dan aturan hubungan antarkolom

Pemeriksaan rentang berikut menggunakan aturan yang masuk akal untuk kolomnya: jumlah harus positif, diskon berada pada 0–100 persen, dan rating layanan berada pada skala 1–5. Nilai `-999` pada rating diperlakukan sebagai placeholder kosong.

Pemeriksaan hubungan jumlah × harga × diskon dibandingkan dengan `total_bayar` bersifat diagnostik. Perbedaan tidak langsung diperbaiki karena dataset bisa memiliki aturan pembulatan atau transaksi khusus yang belum dijelaskan.

In [7]:
# Placeholder rating ditandai kosong, bukan diganti dengan nilai tebakan.
data["rating_layanan"] = data["rating_layanan"].replace(-999, np.nan)

data["flag_jumlah_tidak_valid"] = data["jumlah"].notna() & (data["jumlah"] <= 0)
data["flag_diskon_tidak_valid"] = data["diskon_persen"].notna() & (
    (data["diskon_persen"] < 0) | (data["diskon_persen"] > 100)
)
data["flag_rating_tidak_valid"] = data["rating_layanan"].notna() & (
    ~data["rating_layanan"].between(1, 5)
)

data["estimasi_bayar"] = (
    data["jumlah"] * data["harga_satuan"] *
    (1 - data["diskon_persen"] / 100)
)
data["selisih_bayar"] = data["total_bayar"] - data["estimasi_bayar"]

for nama, kolom in [
    ("Jumlah tidak valid", "flag_jumlah_tidak_valid"),
    ("Diskon di luar 0–100%", "flag_diskon_tidak_valid"),
    ("Rating di luar 1–5", "flag_rating_tidak_valid"),
]:
    print(f"{nama}: {int(data[kolom].sum())} baris")

display(data.loc[data["flag_jumlah_tidak_valid"] | data["flag_diskon_tidak_valid"],
                 ["id_transaksi", "jumlah", "harga_satuan", "diskon_persen", "total_bayar"]].head(20))
print("Transaksi dengan selisih pembayaran lebih dari Rp1:",
      int(data["selisih_bayar"].abs().gt(1).sum()))

Jumlah tidak valid: 29 baris
Diskon di luar 0–100%: 9 baris
Rating di luar 1–5: 0 baris


,id_transaksi,jumlah,harga_satuan,diskon_persen,total_bayar
130,TRX250295,2,5000,-20,10000.0
175,TRX251170,0,4000,0,8000.0
269,TRX250072,-4,4000,0,16000.0
285,TRX250170,0,35000,0,70000.0
296,TRX251459,-4,8000,0,32000.0
302,TRX250753,-8,500,0,4000.0
410,TRX250675,0,500,0,1000.0
432,TRX251255,-6,4000,0,12000.0
453,TRX251414,0,8000,10,21600.0
528,TRX250827,7,6000,-20,42000.0


Transaksi dengan selisih pembayaran lebih dari Rp1: 88


## 8. Nilai kosong yang bergantung pada jenis transaksi

Kolom berat barang tidak selalu relevan untuk semua kategori. Kekosongan `berat_kg` pada kategori jasa atau pulsa dapat bersifat struktural, sehingga tidak tepat diisi menggunakan rata-rata berat produk. Tabel berikut menguji pola kosong berdasarkan kategori.

In [8]:
pola_berat = data.groupby("kategori", dropna=False).agg(
    jumlah_transaksi=("berat_kg", "size"),
    berat_kosong=("berat_kg", lambda s: int(s.isna().sum())),
    persen_kosong=("berat_kg", lambda s: round(s.isna().mean() * 100, 2))
).sort_values("persen_kosong", ascending=False)
display(pola_berat)

,jumlah_transaksi,berat_kosong,persen_kosong
kategori,,,
Jasa,151,151,100.0
Pulsa,148,148,100.0
ATK,323,52,16.1
Makanan,508,0,0.0
Minuman,413,0,0.0


## 9. Ringkasan profil kualitas data

Tabel ini merangkum indikator utama dari data yang telah dinormalisasi. Penanda masalah tetap dipertahankan supaya baris bermasalah dapat ditelusuri, bukan disembunyikan melalui penghapusan otomatis.

In [9]:
profil = pd.DataFrame([
    ["Kelengkapan", "Sel kosong", int(data.isna().sum().sum()), "Dihitung setelah placeholder yang jelas diidentifikasi."],
    ["Keunikan", "ID transaksi kosong", int(data["flag_id_kosong"].sum()), "Perlu konfirmasi sumber data."],
    ["Keunikan", "Baris dengan ID berulang", int(data["flag_id_berulang"].sum()), "Tidak otomatis dihapus."],
    ["Keunikan", "Baris duplikat persis", int(data["flag_duplikat_persis"].sum()), "Perlu pemeriksaan sebelum penghapusan."],
    ["Validitas", "Jumlah tidak positif", int(data["flag_jumlah_tidak_valid"].sum()), "Ditandai untuk investigasi."],
    ["Validitas", "Diskon di luar rentang", int(data["flag_diskon_tidak_valid"].sum()), "Ditandai untuk investigasi."],
    ["Validitas", "Rating di luar skala", int(data["flag_rating_tidak_valid"].sum()), "Rating -999 telah diubah menjadi kosong."],
], columns=["dimensi", "indikator", "jumlah_baris_atau_sel", "catatan"])
display(profil)

,dimensi,indikator,jumlah_baris_atau_sel,catatan
0,Kelengkapan,Sel kosong,1638,Dihitung setelah placeholder yang jelas diiden...
1,Keunikan,ID transaksi kosong,7,Perlu konfirmasi sumber data.
2,Keunikan,Baris dengan ID berulang,85,Tidak otomatis dihapus.
3,Keunikan,Baris duplikat persis,78,Perlu pemeriksaan sebelum penghapusan.
4,Validitas,Jumlah tidak positif,29,Ditandai untuk investigasi.
5,Validitas,Diskon di luar rentang,9,Ditandai untuk investigasi.
6,Validitas,Rating di luar skala,0,Rating -999 telah diubah menjadi kosong.


## 10. Membuat dataset hasil olahan

Untuk menjaga keterlacakan, baris tidak dihapus otomatis pada notebook ini. Kolom `flag_...` membantu menentukan baris mana yang perlu ditinjau. Kolom bantu perhitungan dapat dihapus dari berkas analisis akhir, sedangkan penanda kualitas dipertahankan.

In [10]:
kolom_bantu = ["estimasi_bayar", "selisih_bayar"]
hasil_bersih = data.drop(columns=kolom_bantu).copy()

Path("data/processed").mkdir(parents=True, exist_ok=True)
hasil_bersih.to_csv("data/processed/transaksi_koperasi_2025_diperiksa.csv", index=False)

print("File tersimpan: data/processed/transaksi_koperasi_2025_diperiksa.csv")
print("Dimensi hasil:", hasil_bersih.shape)

File tersimpan: data/processed/transaksi_koperasi_2025_diperiksa.csv
Dimensi hasil: (1543, 22)


## 11. Membuat catatan proses secara otomatis

Catatan ini mencatat jumlah yang terukur dari eksekusi. Kolom alasan tetap perlu dibaca dan disesuaikan jika ditemukan aturan tambahan dari dosen atau pemilik dataset.

In [11]:
jumlah_awal = data_mentah.shape[0]
jumlah_akhir = hasil_bersih.shape[0]
log = f'''# Catatan Pembersihan Data

**Dataset:** transaksi_koperasi_2025.csv
**Baris mentah:** {jumlah_awal} → **baris hasil pemeriksaan:** {jumlah_akhir}

| Tahap | Hasil terukur | Keputusan dan alasan |
|---|---:|---|
| Perapian spasi teks | Diterapkan pada seluruh kolom teks | Mengurangi variasi yang hanya disebabkan spasi luar. |
| Placeholder metode pembayaran | {int(data["metode_bayar"].isna().sum())} nilai kosong pada kolom setelah normalisasi | Label strip/tidak diketahui bukan metode pembayaran yang dapat dianalisis. |
| Konversi tanggal | {int(data["tanggal_transaksi"].isna().sum())} tanggal kosong/gagal dibaca setelah konversi | Nilai gagal tidak ditebak; perlu verifikasi sumber. |
| Konversi total bayar | {int(data["total_bayar"].isna().sum())} pembayaran kosong/gagal dibaca | Format mata uang dibersihkan sebelum konversi numerik. |
| ID transaksi kosong | {int(data["flag_id_kosong"].sum())} baris | Tidak dibuat ID pengganti karena tidak ada sumber pembuktian. |
| ID transaksi berulang | {int(data["flag_id_berulang"].sum())} baris | Tidak dihapus otomatis; ID berulang belum tentu transaksi identik. |
| Rating placeholder -999 | Diubah menjadi kosong | -999 berada di luar skala rating 1–5 dan merupakan kode placeholder pada data ini. |
| Jumlah tidak positif | {int(data["flag_jumlah_tidak_valid"].sum())} baris | Ditandai; koreksi memerlukan bukti transaksi. |
| Diskon di luar 0–100% | {int(data["flag_diskon_tidak_valid"].sum())} baris | Ditandai; tidak dipotong ke rentang valid secara otomatis. |

## Hal yang sengaja tidak diubah
- Baris dengan ID transaksi berulang tidak dihapus sebelum dipastikan apakah benar transaksi ganda.
- Nilai berat yang kosong tidak diisi rata-rata karena berat tidak relevan untuk seluruh kategori.
- Nilai pembayaran yang tidak sesuai estimasi tidak dikoreksi tanpa aturan bisnis yang terverifikasi.

## Pertanyaan untuk konfirmasi
- Apakah satu ID transaksi hanya boleh muncul satu kali?
- Apakah `total_bayar` selalu dihitung dari jumlah, harga satuan, dan diskon?
- Apakah nilai diskon negatif atau di atas 100% mungkin merepresentasikan kode khusus?

## Reproduktibilitas
- [ ] Jalankan ulang notebook dari awal sampai akhir.
- [ ] Tinjau seluruh flag dan keputusan bisnis.
- [ ] Pastikan file mentah tidak tertimpa.
- [ ] Simpan hasil di `data/processed/`.
'''
Path("cleaning_log.md").write_text(log, encoding="utf-8")
print(log)

# Catatan Pembersihan Data

**Dataset:** transaksi_koperasi_2025.csv  
**Baris mentah:** 1543 → **baris hasil pemeriksaan:** 1543

| Tahap | Hasil terukur | Keputusan dan alasan |
|---|---:|---|
| Perapian spasi teks | Diterapkan pada seluruh kolom teks | Mengurangi variasi yang hanya disebabkan spasi luar. |
| Placeholder metode pembayaran | 28 nilai kosong pada kolom setelah normalisasi | Label strip/tidak diketahui bukan metode pembayaran yang dapat dianalisis. |
| Konversi tanggal | 0 tanggal kosong/gagal dibaca setelah konversi | Nilai gagal tidak ditebak; perlu verifikasi sumber. |
| Konversi total bayar | 0 pembayaran kosong/gagal dibaca | Format mata uang dibersihkan sebelum konversi numerik. |
| ID transaksi kosong | 7 baris | Tidak dibuat ID pengganti karena tidak ada sumber pembuktian. |
| ID transaksi berulang | 85 baris | Tidak dihapus otomatis; ID berulang belum tentu transaksi identik. |
| Rating placeholder -999 | Diubah menjadi kosong | -999 berada di luar skala rating

## 12. Interpretasi akhir

Lengkapi bagian ini setelah menjalankan notebook:

1. Masalah kualitas data yang paling berpengaruh terhadap analisis adalah ...
2. Keputusan pembersihan yang paling penting beserta alasannya adalah ...
3. Masalah yang belum dapat diselesaikan tanpa konfirmasi pemilik data adalah ...
4. Keterbatasan hasil analisis ini adalah ...

Tuliskan kesimpulan berdasarkan angka yang benar-benar muncul pada eksekusi notebook, bukan berdasarkan contoh dari sumber lain.